In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version :", tf.__version__)
print("GPU available      :", tf.config.list_physical_devices('GPU'))

In [ ]:
import os

# Find where the splits folder actually is
for root, dirs, files in os.walk('/content'):
    for d in dirs:
        if d == 'train':
            print(os.path.join(root, d))

In [ ]:
import os

TRAIN_DIR = '/content/data/data/splits/train'
VAL_DIR   = '/content/data/data/splits/val'
TEST_DIR  = '/content/data/data/splits/test'

# Verify
for split, split_dir in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
    total = sum(len(os.listdir(os.path.join(split_dir, actor)))
                for actor in os.listdir(split_dir))
    print(f"{split}: {total} images, {len(os.listdir(split_dir))} actors")

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version :", tf.__version__)
print("GPU available      :", tf.config.list_physical_devices('GPU'))

In [ ]:
TRAIN_DIR = '/content/data/data/splits/train'
VAL_DIR   = '/content/data/data/splits/val'
TEST_DIR  = '/content/data/data/splits/test'

IMG_SIZE   = (160, 160)
BATCH_SIZE = 32

# Training — with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1
)

# Validation & test — no augmentation
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"\nClasses found : {train_gen.num_classes}")
print(f"Train batches : {len(train_gen)}")
print(f"Val batches   : {len(val_gen)}")

In [ ]:
NUM_CLASSES = 100

# Load MobileNetV2 — without top classification layer
base_model = MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze ALL layers
base_model.trainable = False

# Build model
inputs  = tf.keras.Input(shape=(160, 160, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

baseline_model = models.Model(inputs, outputs)

baseline_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Count trainable vs frozen parameters
trainable     = sum([tf.size(w).numpy() for w in baseline_model.trainable_weights])
non_trainable = sum([tf.size(w).numpy() for w in baseline_model.non_trainable_weights])

print(f"Trainable parameters     : {trainable:,}")
print(f"Non-trainable parameters : {non_trainable:,}")
print(f"Total parameters         : {trainable + non_trainable:,}")

In [ ]:
import os

SAVE_PATH = '/content/drive/MyDrive/capstone_facerec/'
os.makedirs(SAVE_PATH, exist_ok=True)

callbacks = [
    ModelCheckpoint(
        filepath=SAVE_PATH + 'baseline_best.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        verbose=1
    )
]

history_baseline = baseline_model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=callbacks

)

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    axes[0].plot(history.history['accuracy'],     label='Train accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Val accuracy')
    axes[0].set_title(f'{title} — Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    # Loss
    axes[1].plot(history.history['loss'],     label='Train loss')
    axes[1].plot(history.history['val_loss'], label='Val loss')
    axes[1].set_title(f'{title} — Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(SAVE_PATH + f'{title}_curves.png')
    plt.show()

# Plot baseline results
plot_history(history_baseline, 'Baseline')

# Evaluate on test set
test_loss, test_acc = baseline_model.evaluate(test_gen, verbose=1)
print(f"\nBaseline Test Accuracy : {test_acc*100:.2f}%")
print(f"Baseline Test Loss     : {test_loss:.4f}")

In [ ]:
# Plot training curves
plot_history(history_baseline, 'Baseline')

# Final evaluation on test set
test_loss, test_acc = baseline_model.evaluate(test_gen, verbose=1)
print(f"\nBaseline Test Accuracy : {test_acc*100:.2f}%")
print(f"Baseline Test Loss     : {test_loss:.4f}")